# Week 17: Production Retrieval-Augmented Generation (Advanced RAG) & Vector Search
## Notebook 05: RAG Triad Evaluation: Context Relevance, Groundedness & Faithfulness

### 1. Conceptual Intuition & Real-World Use Case
Deploying a RAG system to production without automated evaluation is dangerous:
- The retriever might return irrelevant noise.
- The LLM might ignore the retrieved context and hallucinate facts.
- The answer might be factually true but fail to address what the user actually asked!

The **RAG Triad** (popularized by TruLens and Ragas) isolates the three points of failure:
1. **Context Relevance:** Are the retrieved chunks relevant to the user query?
2. **Groundedness / Faithfulness:** Is every claim in the generated answer supported strictly by the retrieved context?
3. **Answer Relevance:** Does the final answer address the user query?

---

### 2. Mathematical Foundations: Faithfulness Scoring
Let the generated answer contain $K$ atomic factual claims: $C = {c_1, c_2, \dots, c_K}$.
Let the retrieved context chunks be $D$.

$$\text{Faithfulness Score} = \frac{\sum_{k=1}^K \mathbb{I}(c_k \text{ is entailed by } D)}{K}$$
- Score $= 1.0$: Zero hallucinations. Every claim is verified by the context.
- Score $< 1.0$: Hallucination detected! Flag for review or fallback.

In [ ]:
import re

# Step 1: Define User Query, Retrieved Context, and Candidate Responses
query = "What is the battery capacity and charging time of the Drone-X?"

context = (
    "The Drone-X quadcopter is equipped with a 5000mAh lithium-polymer battery. "
    "Under optimal conditions, the drone achieves a flight time of 32 minutes. "
    "Fast charging allows the battery to reach 80 percent charge in 45 minutes."
)

# Response A: Grounded and faithful
response_faithful = (
    "The Drone-X features a 5000mAh battery. "
    "It supports fast charging to 80% capacity within 45 minutes."
)

# Response B: Hallucinated / Unfaithful
response_hallucinated = (
    "The Drone-X features a 5000mAh battery. "
    "It takes 2 hours to fully charge and includes solar panel charging."
)

# Step 2: Implement Claim Extraction & Verification Engine
def extract_claims(text):
    # Splits response into individual testable sentence claims
    return [s.strip() for s in re.split(r'[.!?]', text) if len(s.strip()) > 5]

def evaluate_faithfulness(response, context):
    claims = extract_claims(response)
    supported_claims = 0
    claim_details = []
    
    context_lower = context.lower()
    for claim in claims:
        # Check token containment and semantic overlap in evidence context
        tokens = [t.lower() for t in claim.split() if len(t) > 3]
        match_count = sum(1 for t in tokens if t in context_lower)
        overlap_ratio = match_count / max(1, len(tokens))
        
        # Rule: A claim is verified if >= 65% of its key terms exist in the context
        is_supported = overlap_ratio >= 0.65
        if is_supported:
            supported_claims += 1
        claim_details.append((claim, is_supported, overlap_ratio))
        
    score = supported_claims / max(1, len(claims))
    return score, claim_details

score_A, details_A = evaluate_faithfulness(response_faithful, context)
score_B, details_B = evaluate_faithfulness(response_hallucinated, context)

print("--- Evaluation Results ---")
print(f"\nResponse A Faithfulness Score: {score_A * 100:.1f}%")
for claim, supp, ratio in details_A:
    status = "VERIFIED" if supp else "HALLUCINATION"
    print(f"  [{status}] ({ratio:.2f} overlap): {claim}")

print(f"\nResponse B Faithfulness Score: {score_B * 100:.1f}%")
for claim, supp, ratio in details_B:
    status = "VERIFIED" if supp else "HALLUCINATION"
    print(f"  [{status}] ({ratio:.2f} overlap): {claim}")

### 3. Step-by-Step Code Walkthrough
1. **Atomic Claim Extraction:** The generated answer is decomposed into distinct verifiable propositions.
2. **Context Entailment Verification:** Each claim is cross-referenced against the retrieved context.
3. **Diagnostic Alert:** Response B is immediately flagged for hallucinating "solar panel charging" and "2 hours", preventing ungrounded output from reaching end users.

---

### 4. Industry Applications & Trade-offs
- **CI/CD Regression Testing:** Automated evaluation suites (Ragas, TruLens) run on pull requests to ensure prompt updates don't degrade retrieval quality.
- **LLM-as-a-Judge:** In production, a smaller evaluator model (e.g. GPT-4o-mini or Claude 3.5 Haiku) is prompted with strict rubric instructions to score faithfulness.

---

### 5. Top Interview Questions & Answers
**Q1: What are the three vertices of the RAG Triad?**
> **Answer:** Context Relevance (query vs retrieved chunks), Groundedness/Faithfulness (retrieved chunks vs generated answer), and Answer Relevance (query vs generated answer). Tracking all three pinpoints whether errors originate from the retriever or the generator.

**Q2: How do you handle non-entailed claims in high-stakes enterprise RAG (e.g. healthcare/finance)?**
> **Answer:** If the faithfulness score falls below a threshold (e.g. $<0.90$), the system triggers an automated fallback: it rejects the answer, informs the user that the retrieved documentation lacks verified facts, and optionally passes the ticket to a human analyst.